In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
class CalciumDecoder:
    def __init__(self, hidden_layer_sizes=(100, 50), random_state=42):
        self.scaler = StandardScaler()
        self.classifier = MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            random_state=random_state,
            max_iter=1000
        )
    
    def get_stimulus_positions(self, pos_matrix):
        """Convert position vectors to stimulus position labels."""
        positions = np.argmax(pos_matrix, axis=0)
        no_stim_mask = ~np.any(pos_matrix, axis=0)
        positions[no_stim_mask] = -1
        return positions
        
    def prepare_data(self, calcium_activity, position_vectors, train_fraction=0.8):
        """
        Prepare the data preserving temporal structure.
        
        Parameters:
        calcium_activity: array of shape (n_timepoints, n_neurons)
        position_vectors: list of 8 arrays indicating stimulus positions
        train_fraction: fraction of data to use for training
        """
        # Get stimulus positions for each timepoint
        positions = self.get_stimulus_positions(position_vectors)
        
        # Keep only timepoints where stimulus was present
        stim_mask = positions != -1
        X = calcium_activity[stim_mask]
        y = positions[stim_mask]
        self.stim_timepoints = np.where(stim_mask)[0]  # Store for plotting
        
        # Split while preserving temporal order
        split_point = int(len(X) * train_fraction)
        X_train = X[:split_point]
        X_test = X[split_point:]
        y_train = y[:split_point]
        y_test = y[split_point:]
        
        # Scale the neural activity data
        X_train = self.scaler.fit_transform(X_train)
        X_test = self.scaler.transform(X_test)
        
        return X_train, X_test, y_train, y_test
    
    def train(self, X_train, y_train):
        """Train the decoder."""
        self.classifier.fit(X_train, y_train)
    
    def predict(self, X):
        """Make predictions on new data."""
        return self.classifier.predict(self.scaler.transform(X))
    
    def evaluate(self, X_test, y_test):
        """Evaluate the decoder's performance."""
        y_pred = self.predict(X_test)
        accuracy = self.classifier.score(self.scaler.transform(X_test), y_test)
        conf_mat = confusion_matrix(y_test, y_pred)
        return accuracy, conf_mat
    
    def plot_confusion_matrix(self, conf_mat):
        """Plot the confusion matrix."""
        plt.figure(figsize=(10, 8))
        sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues')
        plt.title('Confusion Matrix')
        plt.xlabel('Predicted Position')
        plt.ylabel('True Position')
        plt.show()
        
    def plot_temporal_decoding(self, calcium_activity, position_vectors, train_fraction=0.8, window_start=0, window_size=500):
        """
        Plot decoding results in temporal sequence, distinguishing train and test sets.
        
        Parameters:
        calcium_activity: array of shape (n_timepoints, n_neurons)
        position_vectors: list of 8 arrays indicating stimulus positions
        train_fraction: fraction of data used for training
        window_start: start index for plotting window
        window_size: number of timepoints to plot
        """
        # Get true positions for all timepoints
        true_positions = self.get_stimulus_positions(position_vectors)
        
        # Get timepoints with stimuli
        stim_mask = true_positions != -1
        stim_times = np.where(stim_mask)[0]
        
        # Find split point in original time series
        n_stim = len(stim_times)
        split_idx = int(n_stim * train_fraction)
        split_time = stim_times[split_idx]
        
        # Make predictions for timepoints with stimuli
        predictions_stim = self.predict(calcium_activity[stim_mask])
        
        # Create full prediction array
        predictions = np.full_like(true_positions, -1, dtype=float)
        predictions[stim_mask] = predictions_stim
        
        # Ensure window bounds are valid
        window_end = min(window_start + window_size, len(true_positions))
        window_start = min(window_start, len(true_positions) - 1)
        
        # Plot
        plt.figure(figsize=(15, 5))
        
        # Plot true positions
        valid_true = np.ma.masked_where(true_positions[window_start:window_end] == -1, 
                                      true_positions[window_start:window_end])
        plt.plot(range(window_start, window_end), valid_true, 
                'b-', label='True Position', alpha=0.5)
        
        # Plot predicted positions for training set
        train_mask = np.arange(len(true_positions)) <= split_time
        train_mask = train_mask & (predictions != -1)
        train_mask = train_mask[window_start:window_end]
        if np.any(train_mask):
            plt.plot(np.arange(window_start, window_end)[train_mask],
                    predictions[window_start:window_end][train_mask],
                    'g.', label='Training Predictions', markersize=10)
        
        # Plot predicted positions for test set
        test_mask = np.arange(len(true_positions)) > split_time
        test_mask = test_mask & (predictions != -1)
        test_mask = test_mask[window_start:window_end]
        if np.any(test_mask):
            plt.plot(np.arange(window_start, window_end)[test_mask],
                    predictions[window_start:window_end][test_mask],
                    'r.', label='Test Predictions', markersize=10)
        
        # Add vertical line showing train/test split
        if window_start <= split_time <= window_end:
            plt.axvline(x=split_time, color='k', linestyle='--', 
                       label='Train/Test Split')
        
        plt.title('Temporal Decoder Performance')
        plt.xlabel('Time (frames)')
        plt.ylabel('Position')
        plt.legend()
        plt.grid(True)
        plt.ylim(-0.5, 7.5)  # Assuming 8 positions (0-7)
        plt.show()

# [Rest of the code remains the same, just update the demo_decoder function to pass train_fraction]

def demo_decoder(calcium_activity, position_vectors, train_fraction=0.8, window_start=0, window_size=500):
    """
    Demonstrate the usage of the calcium decoder.
    """
    # Initialize the decoder
    decoder = CalciumDecoder()
    
    # Prepare the data
    print("Preparing data...")
    X_train, X_test, y_train, y_test = decoder.prepare_data(
        calcium_activity, 
        position_vectors,
        train_fraction=train_fraction
    )
    
    # Train the decoder
    print("Training decoder...")
    decoder.train(X_train, y_train)
    
    # Evaluate
    accuracy, conf_mat = decoder.evaluate(X_test, y_test)
    print(f"\nTest set accuracy: {accuracy:.2f}")
    
    # Plot confusion matrix
    decoder.plot_confusion_matrix(conf_mat)
    
    # Plot temporal decoding
    decoder.plot_temporal_decoding(calcium_activity, position_vectors,
                                 train_fraction=train_fraction,
                                 window_start=window_start,
                                 window_size=window_size)
    
    # Print detailed classification report
    y_pred = decoder.predict(X_test)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    return decoder

In [ ]:
from pathlib import Path
import flammkuchen as fl

In [ ]:
master =  Path(r"Z:\Hagar and Ot\e0075\habenula")
fish_list = list(master.glob("*_f*"))

n_dirs = 8

In [ ]:
fish = Path(r"Z:\Hagar and Ot\e0075\habenula\240625_f2_h2b-6s_habenula_pre_v06\suite2p\0002")
traces = fl.load(fish / 'filtered_traces.h5')['undetr']
np.shape(traces)

In [ ]:
hab_coords_l = fl.load(fish / 'habenula_coords.h5')['lhab_coords']
hab_coords_r = fl.load(fish / 'habenula_coords.h5')['rhab_coords']

habenula_traces_l = traces[:,hab_coords_l]
habenula_traces_r = traces[:,hab_coords_r]

In [ ]:
regs = fl.load(fish / 'sensory_regressors_cells.h5')['regressors']

In [ ]:
#traces_with_noise = np.copy(traces)
traces_with_noise = np.random.rand(2411,354)

In [ ]:
# Run the decoder
decoder = demo_decoder(habenula_traces_r, regs,
                      window_start=0,    # start frame to plot
                      window_size=2400)   # number of frames to plot

In [ ]:
240/15/8

In [ ]:
# Run the decoder
decoder = demo_decoder(habenula_traces_l, regs)



In [ ]:
def create_shuffled_control(position_vectors):
    """
    Create shuffled version of position vectors that preserves stimulus duration structure.
    
    Parameters:
    position_vectors: list of 8 arrays indicating stimulus positions
    
    Returns:
    shuffled_position_vectors: list of 8 shuffled arrays with same length as input
    """
    # Get original length
    original_length = len(position_vectors[0])
    
    # Stack position vectors into a matrix
    pos_matrix = np.vstack(position_vectors)
    
    # Find boundaries of stimulus presentations
    changes = np.diff(pos_matrix, axis=1)
    change_points = np.where(np.any(changes != 0, axis=0))[0] + 1
    
    # Add start and end points
    block_boundaries = np.concatenate([[0], change_points, [pos_matrix.shape[1]]])
    
    # Extract blocks
    blocks = []
    for i in range(len(block_boundaries) - 1):
        start = block_boundaries[i]
        end = block_boundaries[i + 1]
        block = pos_matrix[:, start:end]
        blocks.append(block)
    
    # Shuffle the blocks
    shuffle_idx = np.random.permutation(len(blocks))
    blocks = [blocks[i] for i in shuffle_idx]
    
    # Reconstruct shuffled position vectors
    shuffled_matrix = np.hstack(blocks)
    
    # Verify length matches original
    assert shuffled_matrix.shape[1] == original_length, \
        "Shuffled data length doesn't match original"
        
    # Convert back to list of vectors
    shuffled_position_vectors = [shuffled_matrix[i, :] for i in range(8)]
    
    return shuffled_position_vectors

def demo_decoder_with_control(calcium_activity, position_vectors, n_shuffles=10, 
                            train_fraction=0.8, window_start=0, window_size=500):
    """
    Demonstrate decoder performance against shuffled controls.
    
    Parameters:
    calcium_activity: array of shape (n_timepoints, n_neurons)
    position_vectors: list of 8 arrays indicating stimulus positions
    n_shuffles: number of shuffled controls to run
    train_fraction: fraction of data to use for training
    window_start: start frame for plotting
    window_size: number of frames to plot
    """
    # Verify input dimensions
    n_timepoints = len(position_vectors[0])
    assert len(calcium_activity) == n_timepoints, \
        f"Calcium activity length ({len(calcium_activity)}) doesn't match position vectors length ({n_timepoints})"
    
    # Run decoder on real data
    print("Running decoder on real data...")
    real_decoder = CalciumDecoder()
    X_train, X_test, y_train, y_test = real_decoder.prepare_data(
        calcium_activity, 
        position_vectors,
        train_fraction=train_fraction
    )
    real_decoder.train(X_train, y_train)
    real_accuracy, real_conf_mat = real_decoder.evaluate(X_test, y_test)
    
    # Run decoders on shuffled controls
    print("\nRunning shuffled controls...")
    shuffle_accuracies = []
    for i in range(n_shuffles):
        print(f"Shuffle {i+1}/{n_shuffles}")
        
        # Create shuffled position vectors
        shuffled_positions = create_shuffled_control(position_vectors)
        
        # Verify shuffled data length
        assert len(shuffled_positions[0]) == n_timepoints, \
            "Shuffled position vectors length doesn't match original"
        
        # Train and evaluate decoder
        shuffle_decoder = CalciumDecoder()
        X_train, X_test, y_train, y_test = shuffle_decoder.prepare_data(
            calcium_activity, 
            shuffled_positions,
            train_fraction=train_fraction
        )
        shuffle_decoder.train(X_train, y_train)
        shuffle_accuracy, _ = shuffle_decoder.evaluate(X_test, y_test)
        shuffle_accuracies.append(shuffle_accuracy)
    
    # Plot results
    plt.figure(figsize=(10, 6))
    plt.hist(shuffle_accuracies, bins=10, alpha=0.5, label='Shuffled Controls')
    plt.axvline(real_accuracy, color='r', linestyle='--', 
                label=f'Real Data (acc={real_accuracy:.3f})')
    plt.xlabel('Accuracy')
    plt.ylabel('Count')
    plt.title('Decoder Performance vs Shuffled Controls')
    plt.legend()
    plt.show()
    
    # Plot example real data decoding
    print("\nPlotting temporal decoding for real data...")
    real_decoder.plot_temporal_decoding(
        calcium_activity, 
        position_vectors,
        train_fraction=train_fraction,
        window_start=window_start,
        window_size=window_size
    )
    
    # Print summary statistics
    print("\nSummary:")
    print(f"Real data accuracy: {real_accuracy:.3f}")
    print(f"Shuffled control accuracy: {np.mean(shuffle_accuracies):.3f} ± {np.std(shuffle_accuracies):.3f}")
    print(f"Shuffle range: [{np.min(shuffle_accuracies):.3f}, {np.max(shuffle_accuracies):.3f}]")
    
    return real_decoder, shuffle_accuracies

In [ ]:
decoder, shuffle_accuracies = demo_decoder_with_control(habenula_traces_r, regs)